# 05 — Pipeline LLM

OpenAI, Anthropic, Gemini, Groq, Cerebras, custom on_message, LLMLoop, tool-call protocol.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
%load_ext autoreload
%autoreload 2

import _setup
env = _setup.load()
print(f'getpatter version target: {env.patter_version}')


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


In [ ]:
import sys
import getpatter
with _setup.cell('version_check', tier=1, env=env) as ok:
    if ok:
        print(f'getpatter {getpatter.__version__} on Python {sys.version.split()[0]}')
        assert getpatter.__version__ >= env.patter_version, \
            f'installed {getpatter.__version__} < target {env.patter_version}'


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


In [ ]:
from getpatter import Patter, Twilio
with _setup.cell('local_mode', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(
                account_sid='ACtest00000000000000000000000000',
                auth_token='test',
            ),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        # Verify carrier is wired up via LocalConfig.
        assert p._local_config.telephony_provider == 'twilio'
        assert p._local_config.phone_number == '+15555550100'
        print(f'provider={p._local_config.telephony_provider}  phone={p._local_config.phone_number}')


### Cloud mode (coming soon)
When `api_key=` is supported, Patter cloud handles telephony. For now, the SDK raises `NotImplementedError` — this cell verifies the guard.


In [ ]:
from getpatter import Patter
with _setup.cell('cloud_mode', tier=1, env=env) as ok:
    if ok:
        try:
            Patter(api_key='pt_test_xxx')
            raise AssertionError('expected NotImplementedError — cloud mode guard missing')
        except NotImplementedError as exc:
            print(f'cloud mode guard OK: {exc}')


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the engine from `engine=` / `stt=`/`tts=`.


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI
with _setup.cell('agent_engines', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid='ACtest00000000000000000000000000',
                           auth_token='test'),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        rt = p.agent(system_prompt='hi', engine=OpenAIRealtime(api_key='sk-test'))
        cv = p.agent(system_prompt='hi', engine=ElevenLabsConvAI(api_key='el-test', agent_id='a1'))
        pl = p.agent(system_prompt='hi')  # default: OpenAI Realtime fallback
        assert rt.provider == 'openai_realtime', rt.provider
        assert cv.provider == 'elevenlabs_convai', cv.provider
        assert pl.provider == 'openai_realtime', pl.provider
        print(f'rt.provider={rt.provider}  cv.provider={cv.provider}  pl.provider={pl.provider}')


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


### Cloud mode
Same SDK, just an `api_key=` instead of a carrier — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the mode from `engine=` / `stt=`/`tts=`.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises LLM provider construction, fallback routing, ChatContext, and (T3) live completions.


### LLM provider construction


In [ ]:
from getpatter import OpenAILLM, AnthropicLLM, GroqLLM, CerebrasLLM, GoogleLLM
with _setup.cell('llm_providers', tier=1, env=env) as ok:
    if ok:
        oai  = OpenAILLM(api_key='sk-test',  model='gpt-4o-mini')
        ant  = AnthropicLLM(api_key='sk-ant-test', model='claude-haiku-4-5-20251001')
        grq  = GroqLLM(api_key='gr-test',   model='llama3-8b-8192')
        cer  = CerebrasLLM(api_key='ce-test', model='llama-4-scout-17b-16e-instruct')
        goo  = GoogleLLM(api_key='go-test',  model='gemini-2.0-flash')
        for name, p in [('OpenAI', oai), ('Anthropic', ant), ('Groq', grq), ('Cerebras', cer), ('Google', goo)]:
            print(f'{name:10s}: model={p.model}')
        assert oai.model == 'gpt-4o-mini'
        assert ant.model == 'claude-haiku-4-5-20251001'


### FallbackLLMProvider


In [ ]:
from getpatter import FallbackLLMProvider, OpenAILLM, AnthropicLLM, AllProvidersFailedError
with _setup.cell('fallback_provider', tier=1, env=env) as ok:
    if ok:
        primary   = OpenAILLM(api_key='sk-test',    model='gpt-4o-mini')
        secondary = AnthropicLLM(api_key='sk-ant-test', model='claude-haiku-4-5-20251001')
        fallback  = FallbackLLMProvider([primary, secondary])
        print(f'FallbackLLMProvider with {len(fallback.providers)} providers:')
        for i, p in enumerate(fallback.providers):
            print(f'  [{i}] {type(p).__name__} model={p.model}')
        print(f'AllProvidersFailedError: {AllProvidersFailedError.__name__}')


### ChatContext


In [ ]:
from getpatter import ChatContext
with _setup.cell('chat_context', tier=1, env=env) as ok:
    if ok:
        ctx = ChatContext(system_prompt='You are a helpful assistant.')
        ctx.add_user('What is the capital of France?')
        ctx.add_assistant('The capital of France is Paris.')
        ctx.add_user('And Germany?')
        msgs = ctx.get_messages()
        print(f'Total messages: {len(msgs)}')
        for m in msgs:
            print(f'  {m.role:9s}: {m.content[:50]}')
        oai_fmt = ctx.to_openai()
        print(f'OpenAI format ({len(oai_fmt)} msgs): roles={[m["role"] for m in oai_fmt]}')
        ant_fmt = ctx.to_anthropic()
        print(f'Anthropic format: {ant_fmt[:1]}')
        assert ctx.length() == 4


### Live: OpenAI chat completion  *(T3 — requires `OPENAI_API_KEY`)*


In [ ]:
import httpx
with _setup.cell('openai_chat_live', tier=3, required=['openai_key'], env=env) as ok:
    if ok:
        resp = httpx.post(
            'https://api.openai.com/v1/chat/completions',
            headers={'Authorization': f'Bearer {env.openai_key}', 'Content-Type': 'application/json'},
            json={
                'model': 'gpt-4o-mini',
                'messages': [{'role': 'user', 'content': 'Reply with exactly: pong'}],
                'max_tokens': 10,
            },
            timeout=30,
        )
        resp.raise_for_status()
        reply = resp.json()['choices'][0]['message']['content']
        print(f'OpenAI reply: {reply!r}')


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Places a call using the Pipeline engine with OpenAI LLM + tool call. Requires `ENABLE_LIVE_CALLS=1`.


### Pre-flight checklist


In [ ]:
with _setup.cell('live_preflight', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env=env) as ok:
    if ok:
        print(f'  carrier:  Twilio {env.twilio_number}  →  {env.target_number}')
        print(f'  LLM:      OpenAI  (gpt-4o-mini)')
        print(f'  webhook:  {env.public_webhook_url or "(ngrok auto-launch)"}')


### Live LLM call with tool *(T4)*


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, tool
with _setup.cell('live_llm_call', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env=env) as ok:
    if ok:
        @tool
        def get_time() -> str:
            """Return the current UTC time."""
            import datetime
            return datetime.datetime.utcnow().strftime('%H:%M UTC')

        p = Patter(
            carrier=Twilio(account_sid=env.twilio_sid, auth_token=env.twilio_token),
            phone_number=env.twilio_number,
            webhook_url=env.public_webhook_url,
        )
        agent = p.agent(
            system_prompt='You are a helpful demo assistant. If asked for the time, use your tool.',
            engine=OpenAIRealtime(api_key=env.openai_key),
            tools=[get_time],
        )
        try:
            await p.call(env.target_number, agent=agent,
                         first_message='Hello! Ask me what time it is.',
                         ring_timeout=env.max_call_seconds)
            print('✓ LLM call with tool completed')
        finally:
            _setup.hangup_leftover_calls(env)
